# Build Japanese–English speech pairs with Qwen

Generate speech from JESC and KFTT and check its quality with Whisper. Select a GPU runtime and follow the cells for your current stage.

| Stage | Cell order |
| --- | --- |
| First run | **1–5 → 6** to set up and test five pairs |
| Start or resume production | **1–5 → 7 → 8** to continue from progress on Drive |
| Export an interim release | **9** |
| Check or export the final release | **10**, after all shards finish |

[Qwen runbook](https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus/blob/main/docs/qwen-runbook.md)


## 1–4. Mount Drive, fetch code, install dependencies, and check the GPU

Run the four code cells below in order. Audio and progress are saved to `MyDrive/ja-en-direct-s2st-corpus-data/`.


In [ ]:
# 1. Mount Google Drive and configure persistent storage.
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

REPO_URL = 'https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus.git'
REPO_DIR = Path('/content/ja-en-direct-s2st-corpus')
DATA_ROOT = Path('/content/drive/MyDrive/ja-en-direct-s2st-corpus-data')
HF_HOME = Path('/content/huggingface')
MODEL_ROOT = Path('/content/s2st-models')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['S2ST_DATA_ROOT'] = str(DATA_ROOT)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['S2ST_MODEL_ROOT'] = str(MODEL_ROOT)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Data:', DATA_ROOT)
print('Local model cache:', HF_HOME)

In [ ]:
# 2. Fetch or update the repository.
import subprocess
import sys
PYTHON = sys.executable

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
CONFIG = REPO_DIR / 'configs' / 'colab-pro.yaml'
print('Revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# 3. Install dependencies in Colab Python 3.13 once per runtime.
subprocess.run([PYTHON, '--version'], check=True)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(
    ['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1', 'sox'],
    check=True,
)
subprocess.run(
    [
        PYTHON, '-m', 'pip', 'install',
        'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ],
    check=True,
)
subprocess.run(
    [PYTHON, '-m', 'pip', 'install', '-r', 'requirements/colab.txt'],
    check=True,
)
subprocess.run(
    [PYTHON, '-m', 'pip', 'install', '--no-deps', '-e', '.'],
    check=True,
)
# Transformers imports optional TorchCodec even for WAV file inputs.
# This pipeline uses FFmpeg; Colab's preinstalled TorchCodec may not match torch.
subprocess.run([PYTHON, '-m', 'pip', 'uninstall', '-y', 'torchcodec'], check=True)
print('Dependencies installed.')

In [ ]:
# 4. Check the GPU. Select a GPU runtime if this check fails.
GPU_CHECK = r'''
import torch
import torchvision
import torchaudio
from transformers import AutoProcessor
from qwen_tts import Qwen3TTSModel
boxes = torch.tensor([[0., 0., 1., 1.]])
assert torchvision.ops.nms(boxes, torch.tensor([1.]), 0.5).tolist() == [0]
print({'torch': torch.__version__, 'torchvision': torchvision.__version__, 'torchaudio': torchaudio.__version__, 'qwen_import': 'ok'}, flush=True)
assert torch.cuda.is_available(), 'No GPU detected. Select a GPU runtime.'
props = torch.cuda.get_device_properties(0)
vram_gib = props.total_memory / 1024**3
dtype = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
print({'gpu': props.name, 'vram_gib': round(vram_gib, 2), 'tts_dtype': dtype})
assert vram_gib >= 14, 'At least 14 GiB of VRAM is required, such as a T4 with about 14.7 GiB.'
'''
result = subprocess.run([PYTHON, '-u', '-c', GPU_CHECK], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout, flush=True)
result.check_returncode()

## 5. Check batch sizes

The defaults for an A100 80 GB are 72 for TTS and 144 for Whisper. On smaller GPUs, the notebook starts with a batch size of one for each.


In [ ]:
# 5. Edit the two values below to set batch sizes.
TTS_BATCH_SIZE = 72
ASR_BATCH_SIZE = 144
import json
gpu = json.loads(subprocess.check_output(
    [PYTHON, '-c', "import json, torch; p = torch.cuda.get_device_properties(0); print(json.dumps({'name': p.name, 'vram_gib': p.total_memory / 1024**3}))"],
    text=True,
))
assert type(TTS_BATCH_SIZE) is int and TTS_BATCH_SIZE > 0
assert type(ASR_BATCH_SIZE) is int and ASR_BATCH_SIZE > 0
# Start with one utterance per batch if the new runtime has a smaller GPU.
if gpu['vram_gib'] < 70:
    TTS_BATCH_SIZE = ASR_BATCH_SIZE = 1
    print('This GPU has less than 70 GiB of VRAM. Both batch sizes are set to 1.')
os.environ['S2ST_TTS_BATCH_SIZE'] = str(TTS_BATCH_SIZE)
os.environ['S2ST_ASR_BATCH_SIZE'] = str(ASR_BATCH_SIZE)
print({**gpu, 'tts_batch_size': TTS_BATCH_SIZE, 'asr_batch_size': ASR_BATCH_SIZE})

## 6. Test five pairs

Generate ten utterances across Japanese and English and run Whisper QC. Review acceptance counts and listen to the audio before starting production.


In [ ]:
CONFIG = REPO_DIR / 'configs' / 'colab-pro.yaml'
# Stream subprocess output and errors to the notebook and save a local copy.
from datetime import datetime
LOG = Path('/content') / ('s2st-smoke-' + datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '.log')
print('Python:', PYTHON, flush=True)
print('Revision:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip(), flush=True)
print('Cache:', os.environ.get('HF_HOME'), 'Models:', os.environ.get('S2ST_MODEL_ROOT'), flush=True)
print('Log:', LOG, flush=True)
with LOG.open('w', encoding='utf-8') as log:
    with subprocess.Popen(
        [PYTHON, '-u', '-m', 's2st_corpus.cli', '--config', str(CONFIG), 'smoke-test'],
        cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
print('returncode:', returncode, flush=True)
if returncode:
    raise RuntimeError(f'Smoke test failed. See the traceback above or {LOG}')

## 7. Check saved progress

Display the completed shard count and the next shard to process.


In [ ]:
# Show saved production progress.
result = subprocess.run(
    [PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG), 'status'],
    cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(result.stdout, flush=True)
result.check_returncode()

## 8. Start or resume production

`RUN_ALL_REMAINING = False` processes the next shard. `True` processes all remaining shards and exports the release at the end. Resume with the same configuration and output directory.


In [ ]:
# True runs all remaining shards. False runs the next unfinished shard.
RUN_ALL_REMAINING = True
COMMAND = 'run-remaining-shards' if RUN_ALL_REMAINING else 'run-next-shard'
from datetime import datetime
LOG_DIR = DATA_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG = LOG_DIR / ('production-' + datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '.log')
print('Log:', LOG, flush=True)
with LOG.open('w', encoding='utf-8') as log:
    with subprocess.Popen(
        [PYTHON, '-u', '-m', 's2st_corpus.cli', '--config', str(CONFIG), COMMAND],
        cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace', bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
print('returncode:', returncode, flush=True)
if returncode:
    raise RuntimeError(f'Production failed. See {LOG}')

## 9. Export an interim release (optional)

Pause production before running this cell. It writes an interim release from completed shards.


In [ ]:
# Optional interim export from completed shards.
subprocess.run(
    [
        PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG),
        'consolidate', '--allow-incomplete',
    ],
    check=True,
)

## 10. Check the final release

Run this cell after all shards finish if the final release still needs to be exported. Check for `complete: true` in `manifests/releases/summary.json`.


In [ ]:
# Final export after all shards finish.
subprocess.run(
    [PYTHON, '-m', 's2st_corpus.cli', '--config', str(CONFIG), 'consolidate'],
    check=True,
)